In [ ]:
import numpy as np


def calculate_distance(x1, y1, x2, y2):
    """
    Euclidean distance between two points.
    """
    return np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)


def calculate_angle(ax, ay, bx, by, cx, cy):
    """
    Calculates the angle ABC (in degrees).
    Returns an angle between 0 and 180 degrees.
    """

    ba = np.array([ax - bx, ay - by])
    bc = np.array([cx - bx, cy - by])

    norm_ba = np.linalg.norm(ba)
    norm_bc = np.linalg.norm(bc)

    if norm_ba == 0 or norm_bc == 0:
        return np.nan

    cosine = np.dot(ba, bc) / (norm_ba * norm_bc)

    # Numerical safety
    cosine = np.clip(cosine, -1.0, 1.0)

    angle = np.degrees(np.arccos(cosine))

    return angle


def calculate_knee_angle(row):
    """
    Hip -> Knee -> Ankle
    """

    return calculate_angle(
        row["hip_x"], row["hip_y"],
        row["knee_x"], row["knee_y"],
        row["ankle_x"], row["ankle_y"]
    )


def calculate_hip_angle(row):
    """
    Shoulder -> Hip -> Knee
    """

    return calculate_angle(
        row["shoulder_x"], row["shoulder_y"],
        row["hip_x"], row["hip_y"],
        row["knee_x"], row["knee_y"]
    )


def calculate_torso_angle(row):
    """
    Angle of the torso relative to vertical.
    """

    dx = row["shoulder_x"] - row["hip_x"]
    dy = row["shoulder_y"] - row["hip_y"]

    angle = np.degrees(np.arctan2(abs(dx), abs(dy)))

    return angle


def calculate_body_height(row):
    """
    Approximate body height using visible body segments.
    """

    torso = calculate_distance(
        row["shoulder_x"], row["shoulder_y"],
        row["hip_x"], row["hip_y"]
    )

    thigh = calculate_distance(
        row["hip_x"], row["hip_y"],
        row["knee_x"], row["knee_y"]
    )

    shin = calculate_distance(
        row["knee_x"], row["knee_y"],
        row["ankle_x"], row["ankle_y"]
    )

    return torso + thigh + shin


def calculate_hip_height(row):
    """
    Returns hip vertical position.
    """

    return row["hip_y"]


def calculate_knee_forward(row):
    """
    Horizontal knee travel relative to the ankle.
    """

    return abs(row["knee_x"] - row["ankle_x"])


def calculate_hip_shift(row):
    """
    Horizontal hip displacement relative to ankle.

    Useful for detecting excessive forward/backward movement.
    """

    return abs(row["hip_x"] - row["ankle_x"])